# 03 · Are the selected genes biologically meaningful?

In [1]:
import os, sys
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
from genepanel.cli import load_config, load_datasets
cfg = load_config('config/default.yaml')
TAB = f"{cfg['results_dir']}/tables"; FIG = f"{cfg['results_dir']}/figures"
pd.set_option('display.precision', 4)

In [2]:
cons = pd.read_csv(f'{TAB}/consensus.csv')
cons[cons.panel == 'consensus10'][['cancer', 'task', 'gene', 'freq']]

,cancer,task,gene,freq
14726,BRCA,stage1_vs_normal,ADAMTS5,0.9667
14727,BRCA,stage1_vs_normal,MMP11,0.9000
14728,BRCA,stage1_vs_normal,FIGF,0.8667
14729,BRCA,stage1_vs_normal,COL10A1,0.8467
14730,BRCA,stage1_vs_normal,LMOD1,0.6267
14731,BRCA,stage1_vs_normal,PPP1R12B,0.5067
14732,BRCA,stage1_vs_normal,NKAPL,0.4400
14733,BRCA,stage1_vs_normal,SDPR,0.4267
14734,BRCA,stage1_vs_normal,LOC572558,0.4067
14735,BRCA,stage1_vs_normal,MME,0.4000


## Overlap with the OncoKB cancer gene list
Hypergeometric test against the filtered gene universe.

In [3]:
pd.read_csv(f'{TAB}/oncokb_overlap.csv').query('n_panel >= 5')

,cancer,task,panel,n_panel,n_hits,expected,p_value,hits,q_value
0,BRCA,stage1_vs_normal,consensus10,10,0,0.6201,1.0000,NaN,1.0
2,BRCA,stage1_vs_normal,lasso@10,6,0,0.3721,1.0000,NaN,1.0
3,BRCA,stage1_vs_normal,lasso@100,37,2,2.2944,0.6777,"APCDD1,FGF1",1.0
5,BRCA,stage1_vs_normal,lasso@20,9,0,0.5581,1.0000,NaN,1.0
7,BRCA,stage1_vs_normal,lasso@50,23,2,1.4262,0.4219,"APCDD1,FGF1",1.0
...,...,...,...,...,...,...,...,...,...
196,LUAD,tumor_vs_normal,variance@100,94,7,5.8200,0.3637,"ASCL1,CPS1,HMGA2,KDM5D,PAX7,SERPINB3,WIF1",1.0
198,LUAD,tumor_vs_normal,variance@20,17,1,1.0526,0.6628,KDM5D,1.0
199,LUAD,tumor_vs_normal,variance@5,5,0,0.3096,1.0000,NaN,1.0
200,LUAD,tumor_vs_normal,variance@50,46,2,2.8481,0.7870,"CPS1,KDM5D",1.0


## Pathway enrichment (Enrichr; background = filtered genes)

In [4]:
enr = pd.read_csv(f'{TAB}/enrichment.csv')
enr.sort_values('Adjusted P-value').groupby(['cancer', 'task']).head(8)[['cancer', 'task', 'panel', 'Gene_set', 'Term', 'Adjusted P-value', 'Genes']]

,cancer,task,panel,Gene_set,Term,Adjusted P-value,Genes
88,BRCA,stage1_vs_normal,variance@100,KEGG_2021_Human,PPAR signaling pathway,6.6215e-09,FABP4;MMP1;FABP7;ADIPOQ;PLIN4;AQP7;HMGCS2;PLIN...
95,BRCA,stage1_vs_normal,variance@100,MSigDB_Hallmark_2020,KRAS Signaling Dn,3.2183e-06,TFAP2B;CPB1;COL2A1;CALML5;KRT15;SLC6A14;GP2;KR...
245,BRCA,tumor_vs_normal,variance@50,MSigDB_Hallmark_2020,KRAS Signaling Dn,3.4934e-06,TFAP2B;COL2A1;CPB1;CALML5;GP2;KRT5;SOX10
236,BRCA,tumor_vs_normal,variance@100,MSigDB_Hallmark_2020,KRAS Signaling Dn,5.3244e-06,TFAP2B;COL2A1;CPB1;CALML5;GP2;KRT5;BMPR1B;SOX1...
237,BRCA,tumor_vs_normal,variance@100,MSigDB_Hallmark_2020,Estrogen Response Late,9.8094e-06,TFF3;AGR2;NPY1R;TFF1;PGR;HMGCS2;DHRS2;KLK11;LTF
55,BRCA,stage1_vs_normal,stability@50,GO_Biological_Process_2023,Extracellular Structure Organization (GO:0043062),2.7157e-05,MMP11;ADAMTS5;IBSP;HPSE2;COL10A1
56,BRCA,stage1_vs_normal,stability@50,GO_Biological_Process_2023,External Encapsulating Structure Organization ...,2.7157e-05,MMP11;ADAMTS5;IBSP;HPSE2;COL10A1
115,BRCA,stage1_vs_normal,variance@50,KEGG_2021_Human,PPAR signaling pathway,4.3237e-05,FABP4;ADIPOQ;HMGCS2;PLIN1;PCK1
208,BRCA,tumor_vs_normal,variance@100,GO_Biological_Process_2023,Intermediate Filament Organization (GO:0045109),4.4729e-05,KRT17;KRT16;KRT14;KRT5;KRT6B;KRT6A
207,BRCA,tumor_vs_normal,variance@100,GO_Biological_Process_2023,Epidermis Development (GO:0008544),4.4729e-05,CALML5;KRT14;KLK5;KRT5;HOXB13;KLK7;S100A7


## Literature support: PubMed co-mentions vs random genes

In [5]:
display(pd.read_csv(f'{TAB}/pubmed_summary.csv'))
pc = pd.read_csv(f'{TAB}/pubmed_counts.csv')
pc[pc.set == 'selected'].sort_values(['cancer', 'task', 'pubmed'], ascending=[True, True, False])

,cancer,task,median_selected,median_random,frac_selected_with_any,frac_random_with_any,mannwhitney_p
0,BRCA,stage1_vs_normal,9.0,1.0,0.9,0.62,0.0133
1,BRCA,tumor_vs_normal,7.5,1.0,0.8,0.62,0.0255
2,LUAD,stage1_vs_normal,2.5,0.0,0.9,0.44,0.0039
3,LUAD,tumor_vs_normal,2.5,0.0,0.8,0.36,0.0014


,cancer,task,gene,set,pubmed
1,BRCA,stage1_vs_normal,MMP11,selected,102
3,BRCA,stage1_vs_normal,COL10A1,selected,34
9,BRCA,stage1_vs_normal,MME,selected,34
2,BRCA,stage1_vs_normal,FIGF,selected,11
7,BRCA,stage1_vs_normal,SDPR,selected,10
0,BRCA,stage1_vs_normal,ADAMTS5,selected,8
6,BRCA,stage1_vs_normal,NKAPL,selected,3
5,BRCA,stage1_vs_normal,PPP1R12B,selected,2
4,BRCA,stage1_vs_normal,LMOD1,selected,1
8,BRCA,stage1_vs_normal,LOC572558,selected,0
